In [ ]:
# (6 points) Explain the difference between tracing-based and scriptingbased JIT compilation. Give a concrete example of a Python function
# that would work correctly with scripting but fail with tracing.

# Tracing Based is: Runs your code, however only records or captures what goes on through-out run-time with your code. Does not capture all components of your code just what is ran/hit through at run-time.

# Scripting Based Jit Compilation is: Reads/runs your code, however this time it captures the entire code. It reads the source code itself and captures all possible components/paths of your code. Regardless if it was hit at run-time or not.

# Concrete Example:

def dynamic_branch(x: torch.Tensor) -> torch.Tensor:
    if x.sum() > 0:
        return x * 2
    else:
        return x / 2

# Tracing is quick, fast and effective too the point, but is only for very specific use cases and statements.

# Scripting is longer, and slower but provides much more detail and scope of the code. What all capabilities are.

In [ ]:
# (6 points) Consider the following JAX function:

import jax
import jax.numpy as jnp

@jax.jit
def process_batch(data, batch_size):
    num_batches = data.shape[0] // batch_size
    results = []
    for i in range(num_batches):
        batch = data[i * batch_size:(i + 1) * batch_size]
        results.append(jnp.sum(batch ** 2))
    return jnp.array(results)


# --- Test it ---
data = jnp.arange(100, dtype=jnp.float32)
output = process_batch(data, batch_size=10)
print(output)

# What potential performance issue might arise with this JIT-compiled function? How would you fix it?

# Your problem with JIT-compiled function is that you are: for i in range(num_batches) you are setting a hard-coded amount of batches. However; if num_batches changes, or your variables change
# You are forcing and requiring JAX to retrace and recompile the entire function from scratch again. Lot's of over-head time and effort to run your function.
# You are wasting a-lot of time waiting for your function to recompile each-time when you use it with different variables and numbers.

# Solution:

@jax.jit
def process_batch_fixed(data, batch_size):
    num_batches = data.shape[0] // batch_size
    data = data[:num_batches * batch_size].reshape(num_batches, batch_size)
    return jax.vmap(lambda batch: jnp.sum(batch ** 2))(data)


# You are now not doing so much recompiling each-time num_batches and or your variables change.
# It adapts better, less wasted time, more working on the goal.

In [3]:
#  (6 points) The lecture mentioned that operator fusion can reduce memory traffic. For the expression y = sin(cos(x)) + exp(x), calculate:

# The number of GPU kernel launches and global memory reads/writes
# in eager mode (assume each operation reads inputs from global memory and writes outputs to global memory)

# Op Code - Reads - Writes
# cos(x) - x - t1
# sin(t1) - t1 - t2
# exp(3) - x - t3
# y - t2,t3 - y

# Problem: We are using global memory, which is slow. The GPU isn't being utilized to its fullest capability.
# We also are reading x twice redundantly.
# Total memory ops: 9

# The number in fused mode (one kernel, intermediate values in registers)

# The theoretical memory bandwidth reduction factor

# Now when we fuse mode one kernel. We are just Reading x and writing to y in one kernel.
# 9 / 2 = 4.5× less global memory traffic

In [ ]:
# (6 points) Why does dynamic control flow (e.g., if statements depending
# on tensor values) pose a challenge for JIT compilation? Explain both the
# JAX and PyTorch approaches to handling this.

# It poses a problem or challenge because since it runs at run-time. If statements or pathways that are undiscovered do not have values. It attempts to build
# the graph ahead of time but with if statements is unable too. Breaking the whole purpose of JIT.

# JIT is great for repeatedly redundant work over-and over again, it does not like surprises such as an If statement.

# JAX: Raises an error or bakes in whichever branch was true at trace-time. The JAX condition becomes an actual node in the graph, and both branches are compiled in, and the correct one runs at runtime.

# PyTorch: Break's the graph. It compiles everything it can up to the problematic if statement point. Then falls back. Graph break occurs.

# JAX: Errors or bakes into one branch, while PyTorch graph breaks entirely and falls back on to Python.


In [ ]:
# (6 points) When you run torch.compile on a function for the first time,
# it’s slower than the eager version. Why? What specific steps happen
# during this first call that don’t happen in subsequent calls?

# First time, Torch.compile is the over-head work/operations.

# Tracing happens,
# Traces both forward and backwards,
# Inductor that takes the graph and optimizes it,
# Kernel compilation. Generate the kernel for your GPU,
# Cache and save the compiled information above for future use.

# Basically that first call, gets your machine warmed up and ready to work. It takes a bit of time to warm up, but once it completes you are ready to go and do not have to repeat it again. Unless you change your code/function.